In [1]:
import igl
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
from scipy.sparse import diags
from src.fatemarkers import FateMarkers
from src import utils 

from meshplot import plot, subplot 
from matplotlib import pyplot as plt

import vtk 
import time 

In [2]:
import importlib

import src.meshharm 
import src.fatemarkers
importlib.reload(src.meshharm)
importlib.reload(src.fatemarkers)
from src.fatemarkers import FateMarkers

In [3]:
path = 'Data/20250714/fractal_output/day1p5/r0.zarr/A/01/0_fused_zillum//fm_data/213'
m = FateMarkers()
m.load_results(path)

coeffs_hks = m.compute_hks_for_new_times(new_ts=[1, 4, 25, 100])

In [3]:
res = np.load('sim/vocab.npz', allow_pickle=True)
vocab = res['vocab']
scaler = res['scalar'].item() 
sigma = res['sigma']
ts = res['ts']

def compute_hks(m): 
    hks = []
    for t in ts:
            x = np.einsum('i, ji->j', np.exp(-m.eigvals*t), m.eigvecs**2)
            hks.append(x) 
    hks = np.array(hks).T 
    return hks

In [18]:
path = 'Data/20250714/fractal_output/day4p5-more/r0.zarr/C/01/0_fused_zillum/meshes/nnorg_linked_annotated/12'
path = 'Data/20250714/fractal_output/day4p5-more/r0.zarr/C/01/0_fused_zillum/meshes/nnorg_linked_annotated/7'

# path = 'Data/20250714/fractal_output/day1p5/r0.zarr/A/01/0_fused_zillum/meshes/nnorg_linked_annotated/213'

m = FateMarkers().load_mesh_from_file(path + '.vtp')
m.align_with_pca()

In [19]:
_ = m.compute_coefficients(lmax=15)

In [16]:
hks = compute_hks(m)
hks_scaled = scaler.transform(hks)
dist = np.linalg.norm(hks_scaled[:, np.newaxis, :]- vocab[np.newaxis, :, :], axis=2)
encoding = np.exp(-dist**2 / (2 * sigma**2))

In [17]:
for i in range(encoding.shape[1]):
    print(np.min(encoding[:, i]), np.max(encoding[:, i]))
    plot(m.v, m.f, c=encoding[:, i])

2.1659e-09 0.0008737252


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.292103…

0.0 0.0


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.292103…

0.0019493763 0.50042474


...

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.292103…

0.0 0.0


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.292103…

0.00023602744 0.99575657


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.292103…

In [21]:
recon_field = m.eigvecs[:, :int(m.lmax**2)] @ m.coeffs_fm[:int(m.lmax**2), ]

for (i, name) in enumerate(m.field_names):
    if i in [4, 6]:
        print(name)
        print(np.max(m.fields[:, i]))
        plot(m.v, m.f, c=m.fields[:, i])
        plot(m.v, m.f, c=recon_field[:, i])

C02.percentile95
11179.0


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.081413…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.081413…

C03.percentile95
489.0


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.081413…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-0.081413…